# ShelfSight — YOLOv8n Shelf Detection Model
### AI Lab Project | CSC-233 | BNU Spring 2026
**Student:** Shameer Nadeem (F2024-0427) | **Role:** Group Lead + Model 1

---

## How to use this notebook
- **First time (already done):** Run all cells top to bottom to train the model
- **After every session restart:** Only run the ONE cell marked 🔄 then skip to Step 11

---

## 🔄 SESSION RESTART? RUN THIS CELL FIRST — THEN SKIP TO STEP 11

In [ ]:
# ═══════════════════════════════════════════════════════
# RUN THIS AFTER EVERY RESTART — defines everything you need
# ═══════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
from pathlib import Path
import yaml, numpy as np, matplotlib.pyplot as plt
from IPython.display import Image as IPImage, display

# ── Key paths ──────────────────────────────────────────
BEST_MODEL  = "/content/drive/MyDrive/ShelfSight/YOLOv8n_Model1-2/weights/best.pt"
RESULTS_DIR = "/content/drive/MyDrive/ShelfSight/YOLOv8n_Model1-2"
IMAGE_SIZE  = 640

# ── Re-download dataset (needed for data.yaml) ─────────
from roboflow import Roboflow
rf = Roboflow(api_key="G2NmxkC2Za5dtJ47ID3a")
project = rf.workspace("rf20-vl").project("soda-bottles-haga")
version = project.version(1)
dataset = version.download("yolov8")

yaml_files = list(Path(".").rglob("data.yaml"))
DATA_YAML  = str(yaml_files[0])

# ── Load trained model ──────────────────────────────────
trained_model = YOLO(BEST_MODEL)

print("=" * 50)
print("Everything loaded. Skip to Step 11 now.")
print("=" * 50)
print(f"Model    : {BEST_MODEL}")
print(f"data.yaml: {DATA_YAML}")
print(f"Classes  : {trained_model.names}")


---
## Phase 1 — Setup
> ⏭️ SKIP if you already ran the restart cell above.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%pip install ultralytics roboflow --quiet
import ultralytics
ultralytics.checks()

In [ ]:
import os, yaml, numpy as np, matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from ultralytics import YOLO
from PIL import Image
from IPython.display import Image as IPImage, display

print("Libraries imported.")

---
## Phase 2 — Download Dataset
> ⏭️ SKIP if you already ran the restart cell above.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="G2NmxkC2Za5dtJ47ID3a")
project = rf.workspace("rf20-vl").project("soda-bottles-haga")
version = project.version(1)
dataset = version.download("yolov8")
print("Downloaded to:", dataset.location)

In [ ]:
yaml_files = list(Path(".").rglob("data.yaml"))
if not yaml_files:
    raise FileNotFoundError("data.yaml not found.")
DATA_YAML = str(yaml_files[0])
print("data.yaml:", DATA_YAML)

with open(DATA_YAML) as f:
    print(f.read())

In [ ]:
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
class_names = cfg.get("names", [])
print("Classes:", class_names)

train_labels = Path(DATA_YAML).parent / "train" / "labels"
class_counts = {i: 0 for i in range(len(class_names))}
if train_labels.exists():
    for lf in train_labels.glob("*.txt"):
        with open(lf) as f:
            for line in f:
                p = line.strip().split()
                if p: class_counts[int(p[0])] += 1

total = sum(class_counts.values())
print("\nClass balance:")
for cid, cnt in class_counts.items():
    name = class_names[cid] if cid < len(class_names) else f"cls{cid}"
    pct  = cnt / total * 100 if total else 0
    print(f"  {name:<20}: {cnt:,}  ({pct:.1f}%)")

fig, ax = plt.subplots(figsize=(6, 3))
names  = [class_names[i] if i < len(class_names) else f"cls{i}" for i in class_counts]
counts = list(class_counts.values())
ax.bar(names, counts, color=["#1D9E75","#378ADD","#EF9F27"])
ax.set_title("Bounding boxes per class")
ax.set_ylabel("Count")
plt.tight_layout(); plt.show()

---
## Phase 3 — Train the Model
> ⛔ SKIP — model already trained and saved to Drive.
> Only run this section if you need to retrain from scratch.

In [ ]:
# ⛔ SKIP THIS CELL — training already done
# Only uncomment and run if retraining from scratch

# IMAGE_SIZE = 640
# BATCH_SIZE = 16
# LR         = 0.01

# model = YOLO("yolov8n.pt")

# results = model.train(
#     data    = DATA_YAML,
#     epochs  = 50,
#     imgsz   = IMAGE_SIZE,
#     batch   = BATCH_SIZE,
#     lr0     = LR,
#     patience= 10,
#     save    = True,
#     plots   = True,
#     project = "/content/drive/MyDrive/ShelfSight",
#     name    = "YOLOv8n_Model1",
# )
# print("Training complete!")

print("Skipped — model already trained.")

---
## Phase 4 — Evaluate the Model
> ✅ Start here after running the 🔄 restart cell at the top.

### Step 11 — Evaluate on the test set

In [ ]:
metrics = trained_model.val(
    data    = DATA_YAML,
    split   = "test",
    imgsz   = IMAGE_SIZE,
    verbose = True,
)

map50     = metrics.box.map50
map5095   = metrics.box.map
precision = metrics.box.mp
recall    = metrics.box.mr

print("\n" + "="*50)
print("MODEL 1 (YOLOv8n) — FINAL RESULTS")
print("="*50)
print(f"  mAP50     : {map50*100:.2f}%")
print(f"  mAP50-95  : {map5095*100:.2f}%")
print(f"  Precision : {precision*100:.2f}%")
print(f"  Recall    : {recall*100:.2f}%")
print("="*50)

if map50 >= 0.70:   grade = "Excellent"
elif map50 >= 0.50: grade = "Good"
elif map50 >= 0.30: grade = "Decent"
else:               grade = "Needs improvement"
print(f"  Grade     : {grade}")

### Step 12 — Plot training curves (loaded from Drive)

In [ ]:
import pandas as pd

results_csv = Path(RESULTS_DIR) / "results.csv"

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    if "metrics/mAP50(B)" in df.columns:
        axes[0].plot(df["metrics/mAP50(B)"], color="#1D9E75", linewidth=2)
        axes[0].set_title("mAP50 (higher = better)")
        axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("mAP50")
        axes[0].grid(alpha=0.3)

    if "train/box_loss" in df.columns:
        axes[1].plot(df["train/box_loss"], label="Train", color="#378ADD", linewidth=2)
        if "val/box_loss" in df.columns:
            axes[1].plot(df["val/box_loss"], label="Val", color="#E24B4A", linewidth=2)
        axes[1].set_title("Box Loss (lower = better)")
        axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

    if "metrics/precision(B)" in df.columns:
        axes[2].plot(df["metrics/precision(B)"], label="Precision", color="#7F77DD", linewidth=2)
        if "metrics/recall(B)" in df.columns:
            axes[2].plot(df["metrics/recall(B)"], label="Recall", color="#EF9F27", linewidth=2)
        axes[2].set_title("Precision & Recall")
        axes[2].set_xlabel("Epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)

    plt.suptitle("YOLOv8n Training Results — Model 1 (Shameer)", fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print(f"results.csv not found at: {results_csv}")
    print("Check your Drive path is correct in the restart cell.")

### Step 13 — Show confusion matrix and plots (from Drive)

In [ ]:
plot_files = [
    f"{RESULTS_DIR}/confusion_matrix.png",
    f"{RESULTS_DIR}/results.png",
    f"{RESULTS_DIR}/PR_curve.png",
    f"{RESULTS_DIR}/F1_curve.png",
]

for plot in plot_files:
    if Path(plot).exists():
        print(f"\n--- {Path(plot).name} ---")
        display(IPImage(plot, width=700))
    else:
        print(f"Not found: {plot}")

---
## Phase 5 — Test on a New Image

### Step 14 — Run detection on a shelf photo
Upload any shelf photo to Colab (folder icon on left → upload) then run this.

In [ ]:
TEST_IMAGE = "test_shelf.jpg"   # <-- change to your uploaded filename

if not Path(TEST_IMAGE).exists():
    print(f"Upload '{TEST_IMAGE}' to Colab first.")
else:
    detect_results = trained_model.predict(
        source  = TEST_IMAGE,
        conf    = 0.5,
        save    = True,
        project = "detections",
        name    = "test1",
    )

    output_imgs = list(Path("detections/test1").glob("*.jpg"))
    if not output_imgs:
        output_imgs = list(Path("detections/test1").glob("*.png"))
    if output_imgs:
        display(IPImage(str(output_imgs[0]), width=700))

    result = detect_results[0]
    print(f"\nDetected {len(result.boxes)} objects:")
    for box in result.boxes:
        name = trained_model.names[int(box.cls)]
        conf = float(box.conf)
        print(f"  {name:<20} confidence: {conf*100:.1f}%")

### Step 15 — Calculate shelf metrics

In [ ]:
if Path(TEST_IMAGE).exists():
    result = detect_results[0]
    boxes  = result.boxes

    if len(boxes) == 0:
        print("No products detected. Lower conf to 0.3 in Step 14.")
    else:
        class_counts = {}
        for box in boxes:
            name = trained_model.names[int(box.cls)]
            class_counts[name] = class_counts.get(name, 0) + 1

        total = sum(class_counts.values())

        print("="*50)
        print("SHELF ANALYSIS RESULTS")
        print("="*50)
        print("Detection counts:")
        for name, count in class_counts.items():
            print(f"  {name:<20}: {count}")

        nestle_key = next((k for k in class_counts if "nestle" in k.lower()), None)
        if nestle_key:
            sos = class_counts[nestle_key] / total * 100
            print(f"\nShare of Shelf (SOS)   : {sos:.1f}%")
            print(f"  Nestle             : {class_counts[nestle_key]}")
            print(f"  Competitor         : {total - class_counts[nestle_key]}")
        else:
            print("\nNote: No 'nestle' class in this dataset.")
            print("All detected classes shown above.")

        print(f"\nTotal products detected: {total}")
        print("="*50)

---
## Phase 6 — Download Files

### Step 16 — Download best.pt from Drive

In [ ]:
from google.colab import files

if Path(BEST_MODEL).exists():
    files.download(BEST_MODEL)
    print("Downloaded:", BEST_MODEL)
else:
    print("File not found:", BEST_MODEL)
    print("Check your Drive path in the restart cell.")

### Step 17 — Download all results as ZIP

In [ ]:
import zipfile

zip_path = "ShelfSight_YOLOv8n_Results.zip"
results_path = Path(RESULTS_DIR)

if results_path.exists():
    with zipfile.ZipFile(zip_path, "w") as zf:
        for f in results_path.rglob("*"):
            if f.is_file():
                zf.write(f, f.relative_to(results_path))
    files.download(zip_path)
    print("Downloaded:", zip_path)
else:
    print("Results folder not found:", RESULTS_DIR)

### Step 18 — Final summary for comparison notebook

In [ ]:
print("=" * 55)
print("SHAMEER — SEND THESE NUMBERS TO COMPARISON NOTEBOOK")
print("=" * 55)
print(f"Model         : YOLOv8n (Model 1)")
print(f"mAP50         : {map50*100:.2f}%")
print(f"mAP50-95      : {map5095*100:.2f}%")
print(f"Precision     : {precision*100:.2f}%")
print(f"Recall        : {recall*100:.2f}%")
print(f"best.pt path  : {BEST_MODEL}")
print("=" * 55)

---
## Notes for the Viva

1. **Why YOLOv8n?** Fastest YOLO version — good baseline to compare all 5 models against. Pre-trained on COCO then fine-tuned on our shelf dataset.

2. **What is mAP50?** Mean Average Precision at 50% IoU. The main accuracy metric for object detection. Above 70% is good for a university project.

3. **Connection to AlexNet lab?** Both use CNN backbones. The AlexNet lab taught us about class imbalance bias — we applied the same check here in Step 5.

4. **Why fine-tuning and not training from scratch?** Training from scratch needs millions of images. Fine-tuning starts from pre-trained weights (edges, shapes already learned) and adapts them to shelf products. Faster and more accurate.

5. **What are the 3 classes in your dataset?** coca-cola, fanta, sprite — soda bottles used as a proxy for Nestlé vs competitor shelf analysis.